# Track A: free Colab data and compute preflight

CPU preparation + free Colab GPU, as confirmed by Daniel on 4 November 2026. This notebook has not been executed in Colab. It checks access and data structure; it does not train a model. GPU availability is inspected at runtime. No paid services are used.

Candidate: Virginia Tech corrosion condition-state dataset, v2 (CC0). Preserve its four semantic classes until the label guidelines are inspected; do not invent a binary Rust mapping. Keep originals, resized copies and crops in the same group. Gate: data/split and endpoint checks precede any model run.

In [ ]:
from pathlib import Path
import sys, json, hashlib, shutil, subprocess
WORK = Path('/content/track-a')
WORK.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Free disk bytes:', shutil.disk_usage(WORK).free)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch unavailable; stop before any model run. CPU metadata checks can continue.')


## Save versioned dataset metadata
Only small public metadata is retrieved here. Download failures are evidence to preserve, not grounds to substitute unverified labels or mirrors.

In [ ]:
import requests, datetime
URL = 'https://api.figshare.com/v2/articles/16624663'
r = requests.get(URL, timeout=30)
r.raise_for_status()
metadata = r.json()
assert metadata['version'] == 2, 'Dataset version changed; inspect before proceeding'
assert metadata['license']['name'] == 'CC0', 'Inspect changed license before proceeding'
metadata_bytes = r.content
path = WORK / 'vt-dataset-v2-metadata.json'
if path.exists():
    raise RuntimeError('Preserve existing capture; choose a new run folder')
path.write_bytes(metadata_bytes)
print('Metadata SHA256:', hashlib.sha256(metadata_bytes).hexdigest())
for item in metadata['files']:
    print(item['id'], item['name'], item['size'], item['download_url'])


## Optional full dataset download
The local access check encountered HTTP 403 after the normal download redirect. Colab access is not verified. Leave the switch False to inspect metadata only. Setting it True downloads approximately 333 MB to temporary Colab storage. This is a data-feasibility check, not permission to run model training. No credentials or access-control bypass are attempted.

In [ ]:
DOWNLOAD_DATASET = False
MAX_BYTES = 350_000_000
archive = WORK / 'vt-corrosion-v2.zip'
if DOWNLOAD_DATASET:
    record = next(x for x in metadata['files'] if x['id'] == 31729733)
    assert record['size'] <= MAX_BYTES
    if archive.exists():
        raise RuntimeError('Preserve existing download; inspect it or choose a new run folder')
    sha = hashlib.sha256()
    legacy_md5 = hashlib.md5()
    total = 0
    partial = archive.with_suffix('.zip.partial')
    try:
        with requests.get(record['download_url'], stream=True, timeout=60) as response:
            response.raise_for_status()
            with partial.open('xb') as stream:
                for chunk in response.iter_content(1024 * 1024):
                    total += len(chunk)
                    if total > MAX_BYTES:
                        raise RuntimeError('Download cap exceeded')
                    sha.update(chunk)
                    legacy_md5.update(chunk)
                    stream.write(chunk)
        assert total == record['size'], 'Size mismatch; keep partial file for inspection'
        expected_md5 = record.get('computed_md5') or record.get('supplied_md5')
        if expected_md5:
            assert legacy_md5.hexdigest().lower() == expected_md5.lower(), 'Repository checksum mismatch'
        partial.rename(archive)
        (WORK / 'download-manifest.json').write_text(json.dumps({
            'source': record['download_url'], 'dataset_version': 2, 'file_id': record['id'],
            'bytes': total, 'sha256': sha.hexdigest(),
            'time_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}, indent=2))
        print('Downloaded:', total, 'bytes; SHA256:', sha.hexdigest())
    except Exception as error:
        (WORK / 'download-failure.json').write_text(json.dumps({
            'url': record['download_url'], 'error': str(error), 'bytes_received': total,
            'time_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}, indent=2))
        raise
else:
    print('Full dataset download disabled; no images or masks acquired.')


## Inventory without extracting or training
Inspect names and mask formats. Filename differences are not independent samples. Bridge/site groups and original-resized equivalences require explicit evidence. This notebook stops before model training until the dated gate permits a bounded pilot.

In [ ]:
import zipfile
if archive.exists():
    with zipfile.ZipFile(archive) as bundle:
        inventory = [{'name': x.filename, 'bytes': x.file_size} for x in bundle.infolist()]
    (WORK / 'archive-inventory.json').write_text(json.dumps(inventory, indent=2))
    print('Entries:', len(inventory))
    for item in inventory[:40]:
        print(item)
else:
    print('No archive yet; preserve the access failure and use the repository download page manually if available.')


## First pilot bounds after data inspection
Proposed ceiling: one seed, compact segmentation model, 256-pixel inputs, mixed precision only on a compatible GPU, at most 5 epochs / 30 minutes. These are resource limits, not power or convergence claims. Exact class mapping, grouping, splits, backbone and pretrained weights must be frozen in the dated pilot decision before implementation. Stop and checkpoint when the runtime ends; never treat an incomplete run as a scientific result.

The repository evaluator and five CPU tests are in `scripts/evaluate_referral.py` and `tests/`. No training code or result is included in this preflight notebook.